# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [1]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Dziwny przypadek psa nocną porą
2. Koncert oratoryjno-pasyjny AMKP
3. Międzynarodowy Dzień Poezji z Krakowem Miastem Literatury UNESCO
4. Śpiewoterapia
5. Czytanie na dywanie
6. Alicja w Krainie Czarów
7. Impro KRK Underground
8. Jestem obok. Wszyscy w domu
9. Bal
10. Amirova Trio & Iwona Karcz-Wojnarowska: Kiedy tradycja spotyka jazz

Czas wykonania: 3.04s


In [28]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Dziwny przypadek psa nocną porą
2. Koncert oratoryjno-pasyjny AMKP
3. Międzynarodowy Dzień Poezji z Krakowem Miastem Literatury UNESCO
4. Śpiewoterapia
5. Czytanie na dywanie
6. Alicja w Krainie Czarów
7. Impro KRK Underground
8. Jestem obok. Wszyscy w domu
9. Bal
10. Amirova Trio & Iwona Karcz-Wojnarowska: Kiedy tradycja spotyka jazz

Czas wykonania (wątki): 0.77s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [20]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [21]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 12 procesach (rdzeniach)...
Zakończono w czasie 0.35s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [12]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

def sequential_fetch(URL):
    print('Rozpoczynam pobieranie sekwencyjne...')
    start = time.time()
    for _ in range(20):
        response = requests.get(URL).json().get('fact')

    print(f"Pobieranie sekwencyjne zakonczone w {time.time() - start:.2f}s\n\n")

def multithreaded_fetch(URL):
    print('Rozpoczynam pobieranie wielowatkowe...')
    start = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
        responses = list(executor.map(lambda _: requests.get(URL).json().get('fact'), range(20)))
    

    print(f"Pobieranie wielowatkowe zakonczone w {time.time() - start:.2f}s\n\n")

sequential_fetch(CAT_API_URL)
multithreaded_fetch(CAT_API_URL)

Rozpoczynam pobieranie sekwencyjne...
Pobieranie sekwencyjne zakonczone w 4.19s


Rozpoczynam pobieranie wielowatkowe...
Pobieranie wielowatkowe zakonczone w 0.42s




### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [11]:
import queue
import threading
import time

TARGET = 20
MAX_GENERATED = 200

even_queue = queue.Queue()
odd_queue = queue.Queue()

processed_lock = threading.Lock()
processed_count = 0
stop_event = threading.Event()


def producer():
    for number in range(1, MAX_GENERATED + 1):
        if stop_event.is_set():
            break

        if number % 2 == 0:
            even_queue.put(number)
        else:
            odd_queue.put(number)

        time.sleep(0.005)

    even_queue.put(None)
    odd_queue.put(None)


def consumer(q, mode):
    global processed_count

    while True:
        item = q.get()

        if item is None:
            break

        with processed_lock:
            if processed_count >= TARGET:
                stop_event.set()
                continue

            processed_count += 1
            print(f"[{mode}] Liczba: {item} (Postęp: {processed_count}/{TARGET})")

            if processed_count >= TARGET:
                stop_event.set()


t_producer = threading.Thread(target=producer)
t_even = threading.Thread(target=consumer, args=(even_queue, "PARZYSTE"))
t_odd = threading.Thread(target=consumer, args=(odd_queue, "NIEPARZYSTE"))

t_producer.start()
t_even.start()
t_odd.start()

t_producer.join()
t_even.join()
t_odd.join()

print("Gotowe!")

[NIEPARZYSTE] Liczba: 1 (Postęp: 1/20)
[PARZYSTE] Liczba: 2 (Postęp: 2/20)
[NIEPARZYSTE] Liczba: 3 (Postęp: 3/20)
[PARZYSTE] Liczba: 4 (Postęp: 4/20)
[NIEPARZYSTE] Liczba: 5 (Postęp: 5/20)
[PARZYSTE] Liczba: 6 (Postęp: 6/20)
[NIEPARZYSTE] Liczba: 7 (Postęp: 7/20)
[PARZYSTE] Liczba: 8 (Postęp: 8/20)
[NIEPARZYSTE] Liczba: 9 (Postęp: 9/20)
[PARZYSTE] Liczba: 10 (Postęp: 10/20)
[NIEPARZYSTE] Liczba: 11 (Postęp: 11/20)
[PARZYSTE] Liczba: 12 (Postęp: 12/20)
[NIEPARZYSTE] Liczba: 13 (Postęp: 13/20)
[PARZYSTE] Liczba: 14 (Postęp: 14/20)
[NIEPARZYSTE] Liczba: 15 (Postęp: 15/20)
[PARZYSTE] Liczba: 16 (Postęp: 16/20)
[NIEPARZYSTE] Liczba: 17 (Postęp: 17/20)
[PARZYSTE] Liczba: 18 (Postęp: 18/20)
[NIEPARZYSTE] Liczba: 19 (Postęp: 19/20)
[PARZYSTE] Liczba: 20 (Postęp: 20/20)
Gotowe!


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [13]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

def run_calculate_power_sums(limit):
    if limit < 1:
        raise ValueError("Podana liczba musi byc wieksza lub rowna 1")

    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()

    chunk = max(1, limit // cores)
    numbers = range(1, limit + 1)

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.map(calculate_power_sum, numbers, chunksize=chunk)

    print(f"Zakonczono w czasie {time.time() - start:.2f}s.")
    print(f"Liczba wynikow: {len(results)}")


if __name__ == "__main__":
    limit = 10_000
    run_calculate_power_sums(limit)

Praca na 8 procesach (rdzeniach)...
Zakonczono w czasie 0.51s.
Liczba wynikow: 10000
